## BASIC STATISTICS-2

#### Problem Statement : Hospital Patient Data Analysis

##### Context:

A hospital maintains patient records including admission details, department, diagnosis, doctor, and bill amount. You have two datasets: one with patient info and another with billing details. Some patients have blank bill amounts, and there are multiple rows for the same patient due to follow-ups.


##### Tasks:

##### 1.	Load the patient dataset and show summary with info().

In [5]:
import pandas as pd

In [6]:
# Load datasets
patients_df = pd.read_csv(r"C:\Users\Rakshitha\Downloads\Patient_Data.csv")
billing_df = pd.read_csv(r"C:\Users\Rakshitha\Downloads\Billing_Data.csv")

patients_df.info()
patients_df.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6 entries, 0 to 5
Data columns (total 7 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   PatientID       6 non-null      int64  
 1   Name            6 non-null      object 
 2   Department      6 non-null      object 
 3   Doctor          6 non-null      object 
 4   BillAmount      4 non-null      float64
 5   ReceptionistID  6 non-null      int64  
 6   CheckInTime     6 non-null      object 
dtypes: float64(1), int64(2), object(4)
memory usage: 468.0+ bytes


,PatientID,Name,Department,Doctor,BillAmount,ReceptionistID,CheckInTime
0,101,Alice,Cardiology,Dr. Smith,5000.0,1,2023-01-10 09:00
1,102,Bob,Neurology,Dr. John,NaN,2,2023-01-11 10:30
2,103,Charlie,Orthopedics,Dr. Lee,7500.0,1,2023-01-12 11:00
3,104,David,Cardiology,Dr. Smith,6200.0,3,2023-01-13 12:00
4,105,Eva,Dermatology,Dr. Rose,NaN,2,2023-01-14 08:45


##### 2.	Select only the columns relevant for billing: ['PatientID', 'Department', 'Doctor', 'BillAmount'].

In [7]:
billing_related = patients_df[['PatientID', 'Department', 'Doctor', 'BillAmount']]
billing_related 

,PatientID,Department,Doctor,BillAmount
0,101,Cardiology,Dr. Smith,5000.0
1,102,Neurology,Dr. John,NaN
2,103,Orthopedics,Dr. Lee,7500.0
3,104,Cardiology,Dr. Smith,6200.0
4,105,Dermatology,Dr. Rose,NaN
5,101,Cardiology,Dr. Smith,5000.0


##### 3.	Drop administrative columns like ['ReceptionistID', 'CheckInTime'].

In [8]:
patients_df = patients_df.drop(columns=['ReceptionistID', 'CheckInTime'], errors='ignore')
patients_df

,PatientID,Name,Department,Doctor,BillAmount
0,101,Alice,Cardiology,Dr. Smith,5000.0
1,102,Bob,Neurology,Dr. John,NaN
2,103,Charlie,Orthopedics,Dr. Lee,7500.0
3,104,David,Cardiology,Dr. Smith,6200.0
4,105,Eva,Dermatology,Dr. Rose,NaN
5,101,Alice,Cardiology,Dr. Smith,5000.0


##### 4.	Use groupby to find total bill amount per department.

In [9]:
dept_bill = patients_df.groupby('Department')['BillAmount'].sum()
print(dept_bill)

Department
Cardiology     16200.0
Dermatology        0.0
Neurology          0.0
Orthopedics     7500.0
Name: BillAmount, dtype: float64


##### 5.	Remove duplicate patient records based on PatientID.

In [10]:
patients_df = patients_df.drop_duplicates(subset='PatientID', keep='first')
patients_df

,PatientID,Name,Department,Doctor,BillAmount
0,101,Alice,Cardiology,Dr. Smith,5000.0
1,102,Bob,Neurology,Dr. John,NaN
2,103,Charlie,Orthopedics,Dr. Lee,7500.0
3,104,David,Cardiology,Dr. Smith,6200.0
4,105,Eva,Dermatology,Dr. Rose,NaN


##### 6.	Fill missing BillAmount values with the mean bill amount.

In [11]:
# Create proper independent DataFrame
patients_df = patients_df[['PatientID', 'Department', 'Doctor', 'BillAmount']].copy()

# Handle missing values
mean_bill = patients_df['BillAmount'].mean()
patients_df['BillAmount'] = patients_df['BillAmount'].fillna(mean_bill)
patients_df

,PatientID,Department,Doctor,BillAmount
0,101,Cardiology,Dr. Smith,5000.000000
1,102,Neurology,Dr. John,6233.333333
2,103,Orthopedics,Dr. Lee,7500.000000
3,104,Cardiology,Dr. Smith,6200.000000
4,105,Dermatology,Dr. Rose,6233.333333


##### 7.	Merge the billing dataset with patient dataset on PatientID.

In [12]:
merged_df = pd.merge(patients_df, billing_df, on='PatientID', how='inner')
merged_df 

,PatientID,Department,Doctor,BillAmount,InsuranceCovered,FinalAmount
0,101,Cardiology,Dr. Smith,5000.000000,2000,3000
1,102,Neurology,Dr. John,6233.333333,1500,3500
2,103,Orthopedics,Dr. Lee,7500.000000,2500,5000
3,104,Cardiology,Dr. Smith,6200.000000,3000,3200
4,105,Dermatology,Dr. Rose,6233.333333,1000,4000


##### 8.	Concatenate an additional DataFrame that contains new patients for the current week (row-wise).

In [13]:
new_patients_df = pd.read_csv(r"C:\Users\Rakshitha\Downloads\Patient_Data.csv")

updated_df = pd.concat([merged_df, new_patients_df], axis=0, ignore_index=True)

##### 9.	Concatenate new billing category columns like ['InsuranceCovered', 'FinalAmount'] (column-wise).

In [14]:
new_columns_df = pd.DataFrame({
    'InsuranceCovered': [True, False, True],  # example values
    'FinalAmount': [5000, 7000, 6500]
})

final_df = pd.concat([updated_df, new_columns_df], axis=1)
final_df

,PatientID,Department,Doctor,BillAmount,InsuranceCovered,FinalAmount,Name,ReceptionistID,CheckInTime,InsuranceCovered,FinalAmount
0,101,Cardiology,Dr. Smith,5000.000000,2000.0,3000.0,NaN,NaN,NaN,True,5000.0
1,102,Neurology,Dr. John,6233.333333,1500.0,3500.0,NaN,NaN,NaN,False,7000.0
2,103,Orthopedics,Dr. Lee,7500.000000,2500.0,5000.0,NaN,NaN,NaN,True,6500.0
3,104,Cardiology,Dr. Smith,6200.000000,3000.0,3200.0,NaN,NaN,NaN,NaN,NaN
4,105,Dermatology,Dr. Rose,6233.333333,1000.0,4000.0,NaN,NaN,NaN,NaN,NaN
5,101,Cardiology,Dr. Smith,5000.000000,NaN,NaN,Alice,1.0,2023-01-10 09:00,NaN,NaN
6,102,Neurology,Dr. John,NaN,NaN,NaN,Bob,2.0,2023-01-11 10:30,NaN,NaN
7,103,Orthopedics,Dr. Lee,7500.000000,NaN,NaN,Charlie,1.0,2023-01-12 11:00,NaN,NaN
8,104,Cardiology,Dr. Smith,6200.000000,NaN,NaN,David,3.0,2023-01-13 12:00,NaN,NaN
9,105,Dermatology,Dr. Rose,NaN,NaN,NaN,Eva,2.0,2023-01-14 08:45,NaN,NaN
